In [0]:
telemetry = spark.table("workspace.default.silver_telemetry")
errors = spark.table("workspace.default.silver_errors")

print("Telemetry Schema:")
telemetry.printSchema()

print("Errors Schema:")
errors.printSchema()

Telemetry Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

Errors Schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- errorID: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

anomaly_data = (
    telemetry
    .select(
        "datetime",
        "machineID",
        "volt",
        "rotate",
        "pressure",
        "vibration"
    )
    .dropna()
)

print("Anomaly feature schema:")
anomaly_data.printSchema()

print("Total records:", anomaly_data.count())

anomaly_data.show(10, truncate=False)

Anomaly feature schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)

Total records: 876100
+-------------------+---------+----------------+----------------+----------------+----------------+
|datetime           |machineID|volt            |rotate          |pressure        |vibration       |
+-------------------+---------+----------------+----------------+----------------+----------------+
|2015-01-01 11:00:00|1        |172.504839196295|430.323362106675|95.9270416939636|35.6550173268837|
|2015-01-01 13:00:00|1        |172.522780814836|409.624717000438|101.00108276407 |35.4820086610704|
|2015-01-02 23:00:00|1        |190.927218387944|437.960441292353|85.2602473493877|40.566201816709 |
|2015-01-09 01:00:00|1        |166.313552313069|426.525911531078|104.548764491032|40.6996365121188|
|2015-01

In [0]:
from pyspark.sql.functions import col

telemetry_with_errors = (
    anomaly_data.alias("t")
    .join(
        errors.select(
            "datetime",
            "machineID",
            "errorID"
        ).alias("e"),
        (col("t.datetime") == col("e.datetime")) &
        (col("t.machineID") == col("e.machineID")),
        "left"
    )
)

print("Telemetry with error-code information:")
telemetry_with_errors.printSchema()

telemetry_with_errors.show(20, truncate=False)

Telemetry with error-code information:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- errorID: string (nullable = true)

+-------------------+---------+----------------+----------------+----------------+----------------+--------+---------+-------+
|datetime           |machineID|volt            |rotate          |pressure        |vibration       |datetime|machineID|errorID|
+-------------------+---------+----------------+----------------+----------------+----------------+--------+---------+-------+
|2015-01-01 11:00:00|1        |172.504839196295|430.323362106675|95.9270416939636|35.6550173268837|NULL    |NULL     |NULL   |
|2015-01-01 13:00:00|1        |172.522780814836|409.624717000438|101.00108276407 |3

In [0]:
anomaly_features = (
    telemetry_with_errors
    .select(
        col("t.datetime").alias("datetime"),
        col("t.machineID").alias("machineID"),
        col("t.volt").alias("volt"),
        col("t.rotate").alias("rotate"),
        col("t.pressure").alias("pressure"),
        col("t.vibration").alias("vibration"),
        col("e.errorID").alias("errorID")
    )
)

print("Anomaly feature dataset schema:")
anomaly_features.printSchema()

print("Total records:", anomaly_features.count())

anomaly_features.show(10, truncate=False)

Anomaly feature dataset schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- errorID: string (nullable = true)

Total records: 876403
+-------------------+---------+----------------+----------------+----------------+----------------+-------+
|datetime           |machineID|volt            |rotate          |pressure        |vibration       |errorID|
+-------------------+---------+----------------+----------------+----------------+----------------+-------+
|2015-01-01 11:00:00|1        |172.504839196295|430.323362106675|95.9270416939636|35.6550173268837|NULL   |
|2015-01-01 13:00:00|1        |172.522780814836|409.624717000438|101.00108276407 |35.4820086610704|NULL   |
|2015-01-02 23:00:00|1        |190.927218387944|437.960441292353|85.2602473493877|40.566201816709 |NULL   |
|2015-01-09 0

In [0]:
from pyspark.sql.functions import first

anomaly_data_clean = (
    anomaly_features
    .groupBy(
        "datetime",
        "machineID",
        "volt",
        "rotate",
        "pressure",
        "vibration"
    )
    .agg(
        first("errorID", ignorenulls=True).alias("errorID")
    )
    .orderBy("machineID", "datetime")
)

print("Clean anomaly dataset schema:")
anomaly_data_clean.printSchema()

print("Total records:", anomaly_data_clean.count())

anomaly_data_clean.show(10, truncate=False)

Clean anomaly dataset schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- errorID: string (nullable = true)

Total records: 876100
+-------------------+---------+----------------+----------------+----------------+----------------+-------+
|datetime           |machineID|volt            |rotate          |pressure        |vibration       |errorID|
+-------------------+---------+----------------+----------------+----------------+----------------+-------+
|2015-01-01 06:00:00|1        |176.217853015625|418.504078221616|113.077935462083|45.0876857639276|NULL   |
|2015-01-01 07:00:00|1        |162.87922289706 |402.747489565395|95.4605253823187|43.4139726834815|NULL   |
|2015-01-01 08:00:00|1        |170.989902405567|527.349825452291|75.2379048586662|34.1788471214451|NULL   |
|2015-01-01 09:

In [0]:
anomaly_pd = (
    anomaly_data_clean
    .select(
        "datetime",
        "machineID",
        "volt",
        "rotate",
        "pressure",
        "vibration",
        "errorID"
    )
    .toPandas()
)

print("Rows:", len(anomaly_pd))
print("Machines:", anomaly_pd["machineID"].nunique())

print("\nSensor features:")
print(
    anomaly_pd[
        ["volt", "rotate", "pressure", "vibration"]
    ].describe()
)

Rows: 876100
Machines: 100

Sensor features:
                volt         rotate       pressure      vibration
count  876100.000000  876100.000000  876100.000000  876100.000000
mean      170.777736     446.605119     100.858668      40.385007
std        15.509114      52.673886      11.048679       5.370361
min        97.333604     138.432075      51.237106      14.877054
25%       160.304927     412.305714      93.498181      36.777299
50%       170.607338     447.558150     100.425559      40.237247
75%       181.004493     482.176600     107.555231      43.784938
max       255.124717     695.020984     185.951998      76.791072


In [0]:
from sklearn.preprocessing import StandardScaler

sensor_columns = [
    "volt",
    "rotate",
    "pressure",
    "vibration"
]

scaler = StandardScaler()

anomaly_pd_scaled = anomaly_pd.copy()

anomaly_pd_scaled[sensor_columns] = scaler.fit_transform(
    anomaly_pd[sensor_columns]
)

print("Standardization completed.")

print("\nScaled sensor features:")
print(
    anomaly_pd_scaled[sensor_columns].describe()
)

Standardization completed.

Scaled sensor features:
               volt        rotate      pressure     vibration
count  8.761000e+05  8.761000e+05  8.761000e+05  8.761000e+05
mean  -4.476882e-17  7.940950e-16  2.854823e-17 -5.138681e-17
std    1.000001e+00  1.000001e+00  1.000001e+00  1.000001e+00
min   -4.735549e+00 -5.850588e+00 -4.491178e+00 -4.749768e+00
25%   -6.752684e-01 -6.511656e-01 -6.661874e-01 -6.717817e-01
50%   -1.098700e-02  1.809305e-02 -3.920012e-02 -2.751397e-02
75%    6.594034e-01  6.753157e-01  6.060966e-01  6.330922e-01
max    5.438546e+00  4.716113e+00  7.701679e+00  6.779077e+00


In [0]:
from sklearn.ensemble import IsolationForest

sensor_data = anomaly_pd_scaled[
    ["volt", "rotate", "pressure", "vibration"]
]

iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.01,
    random_state=42,
    n_jobs=-1
)

iso_forest.fit(sensor_data)

print("Isolation Forest training completed.")

Isolation Forest training completed.


In [0]:
anomaly_pd_scaled["anomaly_score"] = iso_forest.decision_function(
    sensor_data
)

anomaly_pd_scaled["is_anomaly"] = (
    iso_forest.predict(sensor_data) == -1
)

print("Anomaly detection completed.")

print("\nTotal records:", len(anomaly_pd_scaled))
print("Anomalies detected:", anomaly_pd_scaled["is_anomaly"].sum())
print(
    "Normal records:",
    (~anomaly_pd_scaled["is_anomaly"]).sum()
)

Anomaly detection completed.

Total records: 876100
Anomalies detected: 8761
Normal records: 867339


In [0]:
anomalies_no_error = anomaly_pd_scaled[
    (anomaly_pd_scaled["is_anomaly"] == True) &
    (anomaly_pd_scaled["errorID"].isna())
].copy()

print("Total anomalies:", anomaly_pd_scaled["is_anomaly"].sum())
print("Anomalies with NO error code:", len(anomalies_no_error))

print(
    "Anomalies with error code:",
    anomaly_pd_scaled[
        (anomaly_pd_scaled["is_anomaly"] == True) &
        (anomaly_pd_scaled["errorID"].notna())
    ].shape[0]
)

print("\nSample anomalies with NO error code:")
print(
    anomalies_no_error[
        [
            "datetime",
            "machineID",
            "volt",
            "rotate",
            "pressure",
            "vibration",
            "anomaly_score"
        ]
    ].head(10)
)

Total anomalies: 8761
Anomalies with NO error code: 8703
Anomalies with error code: 58

Sample anomalies with NO error code:
                datetime  machineID  ...  vibration  anomaly_score
610  2015-01-26 16:00:00          1  ...   0.083897      -0.003256
611  2015-01-26 17:00:00          1  ...  -2.483436      -0.006775
687  2015-01-29 21:00:00          1  ...   1.709849      -0.012891
822  2015-02-04 12:00:00          1  ...  -2.206393      -0.003606
932  2015-02-09 02:00:00          1  ...  -3.232550      -0.012314
1020 2015-02-12 18:00:00          1  ...  -0.413115      -0.000070
1101 2015-02-16 03:00:00          1  ...   0.100078      -0.024469
1132 2015-02-17 10:00:00          1  ...  -0.091604      -0.014988
1216 2015-02-20 22:00:00          1  ...  -2.682038      -0.058584
1338 2015-02-26 00:00:00          1  ...  -3.299248      -0.007487

[10 rows x 7 columns]


In [0]:
q5_result = anomaly_pd_scaled[
    [
        "datetime",
        "machineID",
        "volt",
        "rotate",
        "pressure",
        "vibration",
        "anomaly_score",
        "is_anomaly",
        "errorID"
    ]
].copy()

q5_result["no_error_anomaly"] = (
    (q5_result["is_anomaly"] == True) &
    (q5_result["errorID"].isna())
)

print("Q5 result created.")
print("Total records:", len(q5_result))
print("Anomalies:", q5_result["is_anomaly"].sum())
print("Anomalies without error:", q5_result["no_error_anomaly"].sum())

print("\nColumns:")
print(q5_result.columns.tolist())

Q5 result created.
Total records: 876100
Anomalies: 8761
Anomalies without error: 8703

Columns:
['datetime', 'machineID', 'volt', 'rotate', 'pressure', 'vibration', 'anomaly_score', 'is_anomaly', 'errorID', 'no_error_anomaly']


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

q5_spark = spark.createDataFrame(q5_result)

gold_table = "workspace.default.gold_q5_multisensor_anomaly"

if spark.catalog.tableExists(gold_table):

    target = DeltaTable.forName(spark, gold_table)

    (
        target.alias("t")
        .merge(
            q5_spark.alias("s"),
            "t.machineID = s.machineID AND t.datetime = s.datetime"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Gold Q5 table updated using MERGE.")

else:

    (
        q5_spark.write
        .format("delta")
        .saveAsTable(gold_table)
    )

    print("Gold Q5 table created.")

print("\nGold Q5 schema:")
spark.table(gold_table).printSchema()

Gold Q5 table created.

Gold Q5 schema:
root
 |-- datetime: timestamp (nullable = true)
 |-- machineID: integer (nullable = true)
 |-- volt: double (nullable = true)
 |-- rotate: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- anomaly_score: double (nullable = true)
 |-- is_anomaly: boolean (nullable = true)
 |-- errorID: string (nullable = true)
 |-- no_error_anomaly: boolean (nullable = true)



In [0]:
gold_q5 = spark.table(
    "workspace.default.gold_q5_multisensor_anomaly"
)

print("Total Gold records:", gold_q5.count())

print("\nAnomaly counts:")
gold_q5.groupBy("is_anomaly").count().orderBy("is_anomaly").show()

print("\nAnomalies without error code:")
gold_q5.filter(
    F.col("no_error_anomaly") == True
).count()

print("\nSample anomalies without error code:")
gold_q5.filter(
    F.col("no_error_anomaly") == True
).orderBy("machineID", "datetime").show(10, truncate=False)

Total Gold records: 876100

Anomaly counts:
+----------+------+
|is_anomaly| count|
+----------+------+
|     false|867339|
|      true|  8761|
+----------+------+


Anomalies without error code:

Sample anomalies without error code:
+-------------------+---------+-------------------+-------------------+-------------------+--------------------+----------------------+----------+-------+----------------+
|datetime           |machineID|volt               |rotate             |pressure           |vibration           |anomaly_score         |is_anomaly|errorID|no_error_anomaly|
+-------------------+---------+-------------------+-------------------+-------------------+--------------------+----------------------+----------+-------+----------------+
|2015-01-26 16:00:00|1        |2.937242684901948  |1.6190108707170054 |-1.4628369578918947|0.0838965509851946  |-0.0032556438914523866|true      |NULL   |true            |
|2015-01-26 17:00:00|1        |2.641469197047888  |0.7430915093521359 |0.40508

In [0]:
anomaly_summary = (
    gold_q5
    .withColumn(
        "anomaly_type",
        F.when(
            (F.col("is_anomaly") == True) &
            F.col("errorID").isNull(),
            "Anomaly - No Error Code"
        )
        .when(
            (F.col("is_anomaly") == True) &
            F.col("errorID").isNotNull(),
            "Anomaly - With Error Code"
        )
        .otherwise("Normal")
    )
    .groupBy("anomaly_type")
    .count()
    .orderBy(F.desc("count"))
)

display(anomaly_summary)

anomaly_type,count
Normal,867339
Anomaly - No Error Code,8703
Anomaly - With Error Code,58


Databricks visualization. Run in Databricks to view.